In [23]:
import pyreadr
import pandas as pd

main_data = pyreadr.read_r("../data/raw/dt_simulated_weekly.RData")
holidays = pyreadr.read_r("../data/raw/dt_prophet_holidays.RData")

"""
# Visualise the raw dictionary structure returned by pyreadr
print(f"Type: {type(main_data)}")
print(f"Number of objects: {len(main_data)}")
print()

for key, value in main_data.items():
    print(f"Key:   '{key}'")
    print(f"Type:  {type(value)}")
    print(f"Shape: {value.shape}  ({value.shape[0]} rows × {value.shape[1]} cols)")
    print(f"Cols:  {value.columns.tolist()}")
    print()
"""

main_data_df = main_data['dt_simulated_weekly']
holidays_df = holidays['dt_prophet_holidays']

In [24]:
# merge main_data_df and holidays_df 
main_data_df['DATE'] = pd.to_datetime(main_data_df['DATE'])
holidays_df['ds'] = pd.to_datetime(holidays_df['ds'])

# min and max dates in main data
print(f"Data covers from {main_data_df['DATE'].dt.year.min()} to {main_data_df['DATE'].dt.year.max()}")

# extract needed rows
de_holidays = holidays_df[(holidays_df['country'] == 'DE') & (holidays_df['year'].between(2015, 2019))]
print("No duplicates") if not de_holidays['ds'].duplicated().any() else print("Has duplicates") #check for duplicates

# account for holidays by week 
de_holidays['week_start'] = de_holidays['ds'] - pd.to_timedelta(de_holidays['ds'].dt.dayofweek, unit='D')
holiday_weekly = de_holidays.groupby('week_start')['holiday'].apply(lambda x: ', '.join(x)).reset_index()

merged_df = main_data_df.merge(
    holiday_weekly[['week_start', 'holiday']], 
    left_on='DATE', 
    right_on='week_start', 
    how='left')

merged_df['is_holiday'] = merged_df['holiday'].notna().astype(int) #create binary column for holiday
merged_df['holiday'] = merged_df['holiday'].fillna('holiday free week') #fill missing values
merged_df = merged_df.drop(columns='week_start') #drop ds

merged_df.shape

Data covers from 2015 to 2019
No duplicates


(208, 14)

In [25]:
print("\nFirst 3 rows:")
merged_df.head(3)


First 3 rows:


,DATE,revenue,tv_S,ooh_S,print_S,facebook_I,search_clicks_P,search_S,competitor_sales_B,facebook_S,events,newsletter,holiday,is_holiday
0,2015-11-23,2.754372e+06,22358.346667,0.0,12728.488889,2.430128e+07,0.000000,0.000000,8125009,7607.132915,na,19401.653846,holiday free week,0
1,2015-11-30,2.584277e+06,28613.453333,0.0,0.000000,5.527033e+06,9837.238486,4133.333333,7901549,1141.952450,na,14791.000000,holiday free week,0
2,2015-12-07,2.547387e+06,0.000000,132278.4,453.866667,1.665159e+07,12044.119653,3786.666667,8300197,4256.375378,na,14544.000000,holiday free week,0


In [26]:
print("\nData types:")
print(merged_df.dtypes)


Data types:
DATE                  datetime64[s]
revenue                     float64
tv_S                        float64
ooh_S                       float64
print_S                     float64
facebook_I                  float64
search_clicks_P             float64
search_S                    float64
competitor_sales_B            int32
facebook_S                  float64
events                          str
newsletter                  float64
holiday                         str
is_holiday                    int64
dtype: object


To breakdown what each column represents:

`tv_S`, `ooh_S`, `print_S`, `facebook_S`, `search_S` are the **media spend** columns. It how much money was spent on each channel per week.

`facebook_I` counts how many times Facebook ads were shown to people that week. It's like the eyeballs of the ads that takes note of the people that saw it.

`search_clicks_P` counts how many times people actually clicked on a paid search ad that week. It measures engagement, not cost.

`revenue` is the **target variable**. Its the weekly sales revenue we are trying to explain and predict.

`competitor_sales_B` tracks how well competitors sold that week. It is used as a control variable to account for market-wide effects.

`events` flags whether a special event happened that week. It is not clear yet whether this refers to a paid marketing promotion or a natural seasonal event, so we will investigate further.

`newsletter` records how many newsletter emails were sent out that week. This is an organic, non-paid channel.

`holiday` is the name of the public holiday that fell in that week, if any. Weeks with no public holiday will show as NaN.

`is_holiday` is a binary column derived from holiday. It is 1 if a public holiday occurred that week and 0 if not.

In [29]:
if merged_df.isnull().sum().sum() > 0:
    print("Dataset has null values")
else:
    print("Dataset doesn't have null values")

Dataset doesn't have null values


In [30]:
# Save as CSV for all future notebooks
merged_df.to_csv("../data/raw/dt_simulated_weekly.csv", index=False)
print("\nSaved to data/raw/dt_simulated_weekly.csv")


Saved to data/raw/dt_simulated_weekly.csv
